# 🖼️ 이미지 & 멀티모달 벡터 인덱싱

이 노트북에서는 기존 인덱스에 **3개의 새로운 필드**를 추가하고 데이터를 채웁니다.

## 📋 새로 추가할 필드

| 필드 | 타입 | 차원 | 소스 | 설명 |
|------|------|------|------|------|
| `image_vector` | Vector | 1024D | 이미지 → **Azure AI Vision** 직접 임베딩 | 순수 이미지 벡터 |
| `structured_features` | Complex | - | 이미지 → **GPT-5.2** → 9개 피처 JSON | 구조화된 제품 특징 |
| `mm_vision_text_vector` | Vector | 1024D | GPT Vision 분석 텍스트 → **Azure AI Vision** 텍스트 임베딩 | 텍스트를 Vision 모델로 벡터화 |

## 🔑 기존 벡터 필드와의 비교

| 필드 | 차원 | 임베딩 모델 | 소스 |
|------|------|------------|------|
| `content_vector` (기존) | 3072D | text-embedding-3-large | 제품메타 + GPT Vision 텍스트 |
| `image_vector` (신규) | 1024D | **Azure AI Vision (Florence)** | 이미지 픽셀 직접 |
| `mm_vision_text_vector` (신규) | 1024D | **Azure AI Vision (Florence)** | GPT Vision 분석 텍스트 |

> 💡 `content_vector`와 `mm_vision_text_vector`는 **같은 텍스트**를 다른 임베딩 모델로 벡터화한 것입니다.  
> 이를 통해 **동일 소스, 다른 모델**의 검색 성능 차이를 비교할 수 있습니다.

---

## 1️⃣ 환경 설정

In [ ]:
import os
import sys
import time
import json
import requests
import importlib
import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchField,
    SearchFieldDataType,
)
from openai import AzureOpenAI

# 프로젝트 루트를 path에 추가
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ""))

# prompts 모듈 reload (파일 수정 후 커널 재시작 없이 반영)
import prompts
importlib.reload(prompts)
from prompts import (
    IMAGE_ANALYSIS_SYSTEM_PROMPT,
    IMAGE_ANALYSIS_USER_PROMPT_INDEXING,
    IMAGE_FEATURE_EXTRACTION_SYSTEM_PROMPT,
    IMAGE_FEATURE_EXTRACTION_USER_PROMPT,
    FEATURE_EXTRACTION_SCHEMA,
)

# 환경 변수 로드
load_dotenv(override=True)

# Azure AI Search 설정
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
SEARCH_ADMIN_KEY = os.getenv("SEARCH_ADMIN_KEY")
INDEX_NAME = os.getenv("SEARCH_INDEX_NAME", "products-index")

# Azure AI Vision 설정 (이미지/텍스트 직접 임베딩)
VISION_ENDPOINT = os.getenv("AZURE_AI_VISION_ENDPOINT").rstrip("/")
VISION_KEY = os.getenv("AZURE_AI_VISION_KEY")

# Azure OpenAI 설정 (GPT Vision + structured features)
OPENAI_ENDPOINT = os.getenv("AZURE_OPEN_AI_ENDPOINT")
OPENAI_KEY = os.getenv("AZURE_OPEN_AI_KEY")
CHAT_DEPLOYMENT = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")
API_VERSION = os.getenv("AZURE_OPENAI_EMBEDDING_API_VERSION")

credential = AzureKeyCredential(SEARCH_ADMIN_KEY)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=credential)
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=credential)
openai_client = AzureOpenAI(azure_endpoint=OPENAI_ENDPOINT, api_key=OPENAI_KEY, api_version=API_VERSION)

print("✅ 환경 설정 완료")
print(f"   Index: {INDEX_NAME}")
print(f"   Vision Endpoint: {VISION_ENDPOINT}")
print(f"   GPT Model: {CHAT_DEPLOYMENT}")

---
## 2️⃣ 인덱스에 3개 신규 필드 추가

| 필드 | 타입 | 설명 |
|------|------|------|
| `image_vector` | Vector (1024D) | 이미지 직접 임베딩 (Azure AI Vision) |
| `structured_features` | Complex (9 sub-fields) | GPT-5.2로 추출한 구조화된 제품 피처 |
| `mm_vision_text_vector` | Vector (1024D) | GPT Vision 텍스트를 Azure AI Vision으로 임베딩 |

In [ ]:
# 기존 인덱스 가져오기
index = index_client.get_index(INDEX_NAME)
existing_field_names = [f.name for f in index.fields]

print(f"✅ 기존 인덱스 로드: {INDEX_NAME}")
print(f"   기존 필드: {existing_field_names}")

fields_added = []

# ── 1) image_vector 필드 (1024D) ──
if "image_vector" not in existing_field_names:
    index.fields.append(SearchField(
        name="image_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        retrievable=True,
        vector_search_dimensions=1024,
        vector_search_profile_name="hnsw-profile"
    ))
    fields_added.append("image_vector (1024D, 이미지 직접 임베딩)")
else:
    print(f"ℹ️  image_vector 필드 이미 존재")

# ── 2) structured_features 필드 (Complex) ──
if "structured_features" not in existing_field_names:
    index.fields.append(SearchField(
        name="structured_features",
        type=SearchFieldDataType.ComplexType,
        fields=[
            SearchField(name="brand", type=SearchFieldDataType.String,
                        filterable=True, searchable=True),
            SearchField(name="model_gender", type=SearchFieldDataType.String,
                        filterable=True, searchable=True),
            SearchField(name="product_type", type=SearchFieldDataType.String,
                        filterable=True, searchable=True),
            SearchField(name="color", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                        filterable=True, searchable=True),
            SearchField(name="material", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                        filterable=True, searchable=True),
            SearchField(name="style", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                        filterable=True, searchable=True),
            SearchField(name="use_case", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                        filterable=True, searchable=True),
            SearchField(name="target_customer", type=SearchFieldDataType.String,
                        filterable=True, searchable=True),
            SearchField(name="feature", type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                        filterable=True, searchable=True),
        ]
    ))
    fields_added.append("structured_features (Complex, 9 sub-fields)")
else:
    print(f"ℹ️  structured_features 필드 이미 존재")

# ── 3) mm_vision_text_vector 필드 (1024D) ──
if "mm_vision_text_vector" not in existing_field_names:
    index.fields.append(SearchField(
        name="mm_vision_text_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True,
        retrievable=True,
        vector_search_dimensions=1024,
        vector_search_profile_name="hnsw-profile"
    ))
    fields_added.append("mm_vision_text_vector (1024D, 텍스트→Vision 임베딩)")
else:
    print(f"ℹ️  mm_vision_text_vector 필드 이미 존재")

# 인덱스 업데이트
if fields_added:
    result = index_client.create_or_update_index(index)
    print(f"\n✅ 필드 추가 완료!")
    for f in fields_added:
        print(f"   + {f}")
    print(f"   총 필드 수: {len(result.fields)}")
    
    # 벡터 필드 확인
    for f in result.fields:
        dim = getattr(f, 'vector_search_dimensions', None)
        if dim:
            print(f"   🔹 {f.name}: {dim}D")
else:
    print("\nℹ️  추가할 필드가 없습니다.")

---
## 3️⃣ 데이터 로드 및 함수 정의

3가지 API를 사용합니다:

| API | 용도 |
|-----|------|
| **Azure AI Vision** `vectorizeImage` | 이미지 URL → 1024D 벡터 |
| **Azure AI Vision** `vectorizeText` | 텍스트 → 1024D 벡터 |
| **GPT-5.2** (Structured Output) | 이미지 → 9개 피처 JSON |

In [ ]:
# CSV 로드
df = pd.read_csv("../00-data/sample_data.csv")
print(f"✅ 데이터 로드: {len(df)}개 제품")
print(f"   이미지 URL 샘플: {df.iloc[0]['image_link']}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# A. Azure AI Vision: 이미지 직접 임베딩 (image_vector용)
# ═══════════════════════════════════════════════════════════
def vectorize_image_url(image_url, max_retries=3):
    """이미지 URL → 1024D 벡터 (Azure AI Vision Florence)"""
    api_url = f"{VISION_ENDPOINT}/computervision/retrieval:vectorizeImage?api-version=2024-02-01&model-version=2023-04-15"
    headers = {"Ocp-Apim-Subscription-Key": VISION_KEY, "Content-Type": "application/json"}
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(api_url, headers=headers, json={"url": image_url}, timeout=30)
            resp.raise_for_status()
            return resp.json().get("vector")
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
            else:
                print(f"❌ 이미지 임베딩 실패: {str(e)[:80]}")
                return None


# ═══════════════════════════════════════════════════════════
# B. Azure AI Vision: 텍스트 임베딩 (mm_vision_text_vector용)
# ═══════════════════════════════════════════════════════════
def vectorize_text_vision(text, max_retries=3):
    """텍스트 → 1024D 벡터 (Azure AI Vision Florence)"""
    api_url = f"{VISION_ENDPOINT}/computervision/retrieval:vectorizeText?api-version=2024-02-01&model-version=2023-04-15"
    headers = {"Ocp-Apim-Subscription-Key": VISION_KEY, "Content-Type": "application/json"}
    
    for attempt in range(max_retries):
        try:
            resp = requests.post(api_url, headers=headers, json={"text": text}, timeout=30)
            resp.raise_for_status()
            return resp.json().get("vector")
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
            else:
                print(f"❌ 텍스트 임베딩 실패: {str(e)[:80]}")
                return None


# ═══════════════════════════════════════════════════════════
# C. GPT-5.2: Structured Features 추출 (structured_features용)
#    스키마: prompts.py → FEATURE_EXTRACTION_SCHEMA
#    프롬프트: prompts.py → IMAGE_FEATURE_EXTRACTION_*
# ═══════════════════════════════════════════════════════════
def extract_structured_features(image_url, title, category, max_retries=3):
    """이미지 + 메타 → GPT-5.2 Structured Output → 9개 피처 JSON"""
    user_prompt = IMAGE_FEATURE_EXTRACTION_USER_PROMPT.format(title=title, category=category)
    
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.create(
                model=CHAT_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": IMAGE_FEATURE_EXTRACTION_SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "text", "text": user_prompt},
                        {"type": "image_url", "image_url": {"url": image_url, "detail": "low"}}
                    ]}
                ],
                response_format=FEATURE_EXTRACTION_SCHEMA,
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
            else:
                print(f"❌ 피처 추출 실패: {str(e)[:80]}")
                return None


# ═══════════════════════════════════════════════════════════
# D. GPT Vision 텍스트 분석 + enriched_content 조합
#    프롬프트: prompts.py → IMAGE_ANALYSIS_*
#    (01-enrich_with_vision과 동일 기준)
# ═══════════════════════════════════════════════════════════
def analyze_image_for_text(image_url, title, category, max_retries=3):
    """이미지 → GPT Vision → 텍스트 설명"""
    user_prompt = IMAGE_ANALYSIS_USER_PROMPT_INDEXING.format(title=title, category=category)
    
    for attempt in range(max_retries):
        try:
            response = openai_client.chat.completions.create(
                model=CHAT_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": IMAGE_ANALYSIS_SYSTEM_PROMPT},
                    {"role": "user", "content": [
                        {"type": "text", "text": user_prompt},
                        {"type": "image_url", "image_url": {"url": image_url, "detail": "low"}}
                    ]}
                ],
            )
            result_text = response.choices[0].message.content.strip()
            for line in result_text.split("\n"):
                if line.strip().startswith("설명:"):
                    return line.strip().replace("설명:", "").strip()
            return result_text
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
            else:
                print(f"❌ 텍스트 분석 실패: {str(e)[:80]}")
                return None


def create_enriched_content(row, image_analysis):
    """
    01-enrich_with_vision과 동일한 텍스트 조합
    제품명 + 브랜드 + 카테고리 + 이미지분석 → enriched_content
    """
    content_parts = []
    if pd.notna(row['title']) and str(row['title']).strip():
        content_parts.append(f"**제품명**: {row['title']}")
    if pd.notna(row['brand']) and str(row['brand']).strip():
        content_parts.append(f"**브랜드**: {row['brand']}")
    if pd.notna(row['category']) and str(row['category']).strip():
        content_parts.append(f"**카테고리**: {row['category']}")
    if image_analysis and isinstance(image_analysis, str) and image_analysis.strip():
        content_parts.append(f"**제품특징**: {image_analysis}")
    return "\n".join(content_parts) if content_parts else "정보 없음"


print("✅ 함수 정의 완료 (프롬프트 & 스키마: prompts.py에서 import)")
print("   A. vectorize_image_url:       이미지 → 1024D (AI Vision)")
print("   B. vectorize_text_vision:     텍스트 → 1024D (AI Vision)")
print("   C. extract_structured_features: 이미지 → 9개 피처 JSON (GPT-5.2)")
print("   D. analyze_image_for_text:    이미지 → 텍스트 설명 (GPT Vision)")
print("      create_enriched_content:   텍스트 조합 (01-enrich_with_vision 동일)")

---
## 4️⃣ API 테스트

샘플 1개로 3개 파이프라인을 모두 검증합니다.

In [ ]:
test_row = df.iloc[0]
print(f"🧪 테스트: {test_row['title'][:50]}...")
print(f"   이미지: {test_row['image_link']}")

# ── 1) image_vector 테스트 ──
print(f"\n{'─'*60}")
print("1️⃣ image_vector (이미지 → AI Vision 임베딩)")
test_img_vec = vectorize_image_url(test_row['image_link'])
if test_img_vec:
    print(f"   ✅ {len(test_img_vec)}D [{test_img_vec[0]:.4f}, {test_img_vec[1]:.4f}, ...]")
else:
    print("   ❌ 실패")

# ── 2) structured_features 테스트 ──
print(f"\n{'─'*60}")
print("2️⃣ structured_features (이미지 → GPT-5.2 → JSON)")
test_features = extract_structured_features(test_row['image_link'], test_row['title'], test_row['category'])
if test_features:
    print(f"   ✅ 피처 추출 성공")
    for k, v in test_features.items():
        print(f"      {k}: {v}")
else:
    print("   ❌ 실패")

# ── 3) mm_vision_text_vector 테스트 ──
# enriched_content = 제품명+브랜드+카테고리+이미지분석 (07-enrich_with_vision과 동일)
print(f"\n{'─'*60}")
print("3️⃣ mm_vision_text_vector (이미지 → GPT Vision 텍스트 → enriched_content → AI Vision 임베딩)")
test_analysis = analyze_image_for_text(test_row['image_link'], test_row['title'], test_row['category'])
if test_analysis:
    test_enriched = create_enriched_content(test_row, test_analysis)
    print(f"   📝 enriched_content:")
    print(f"   {'='*60}")
    print(f"   {test_enriched[:300]}...")
    print(f"   {'='*60}")
    test_text_vec = vectorize_text_vision(test_enriched)
    if test_text_vec:
        print(f"   ✅ {len(test_text_vec)}D [{test_text_vec[0]:.4f}, {test_text_vec[1]:.4f}, ...]")
    else:
        print("   ❌ Vision 텍스트 임베딩 실패")
else:
    print("   ❌ GPT Vision 분석 실패")

---
## 5️⃣ Step A: image_vector 생성

247개 이미지를 Azure AI Vision `vectorizeImage` API로 직접 1024D 벡터로 변환합니다.

```
이미지 URL → Azure AI Vision Florence → 1024D 벡터
```

In [ ]:
print(f"🚀 Step A: image_vector 생성 ({len(df)}개)\n")

image_vectors = []
img_ok, img_fail = 0, 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="A. 이미지 벡터화"):
    vec = vectorize_image_url(row['image_link'])
    image_vectors.append(vec)
    if vec:
        img_ok += 1
    else:
        img_fail += 1
    time.sleep(0.2)

df['image_vector'] = image_vectors

print(f"\n✅ Step A 완료: 성공 {img_ok}개 / 실패 {img_fail}개")
if img_ok > 0:
    first_valid = next(v for v in image_vectors if v is not None)
    print(f"   차원: {len(first_valid)}D")

In [ ]:
# Step A 검증
print("📊 Step A 검증: image_vector 샘플")
print("=" * 70)
for i in range(min(5, len(df))):
    vec = df.iloc[i]['image_vector']
    status = f"✅ {len(vec)}D" if vec else "❌"
    print(f"  {i+1}. {df.iloc[i]['title'][:45]:<47} {status}")
print("=" * 70)

---
## 6️⃣ Step B: structured_features 생성

GPT-5.2 Structured Output으로 이미지에서 9개 피처를 JSON으로 추출합니다.

```
이미지 + 제품명 + 카테고리 → GPT-5.2 → structured JSON (9 fields)
```

In [ ]:
print(f"🚀 Step B: structured_features 생성 ({len(df)}개)\n")

all_features = []
feat_ok, feat_fail = 0, 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="B. 피처 추출"):
    features = extract_structured_features(row['image_link'], row['title'], row['category'])
    all_features.append(features)
    if features:
        feat_ok += 1
    else:
        feat_fail += 1
    time.sleep(0.5)

df['structured_features'] = all_features

print(f"\n✅ Step B 완료: 성공 {feat_ok}개 / 실패 {feat_fail}개")

In [ ]:
# Step B 검증
print("📊 Step B 검증: structured_features 샘플")
print("=" * 80)
for i in range(min(3, len(df))):
    feat = df.iloc[i]['structured_features']
    print(f"\n{i+1}. {df.iloc[i]['title'][:45]}...")
    if feat:
        print(f"   ✅ product_type: {feat.get('product_type', '-')}")
        print(f"      color: {feat.get('color', [])}")
        print(f"      material: {feat.get('material', [])}")
        print(f"      style: {feat.get('style', [])}")
        print(f"      use_case: {feat.get('use_case', [])}")
        print(f"      target_customer: {feat.get('target_customer', '-')}")
        print(f"      feature: {feat.get('feature', [])}")
    else:
        print(f"   ❌ 추출 실패")
print("\n" + "=" * 80)

---
## 7️⃣ Step C: mm_vision_text_vector 생성

01-enrich_with_vision과 **동일한 텍스트 조합**을 만들고, Azure AI Vision `vectorizeText`로 임베딩합니다.

```
이미지 → GPT Vision → 텍스트 설명 추출
    ↓
제품명 + 브랜드 + 카테고리 + 텍스트 설명 = enriched_content (01-enrich_with_vision 동일)
    ↓
enriched_content → Azure AI Vision vectorizeText → 1024D 벡터
```

> 💡 `content_vector`(기존)와의 차이: 같은 텍스트를 text-embedding-3-large(3072D) 대신 **AI Vision(1024D)**으로 임베딩

In [ ]:
print(f"🚀 Step C: mm_vision_text_vector 생성 ({len(df)}개)")
print("   C-1: GPT Vision으로 텍스트 설명 추출")
print("   C-2: enriched_content 텍스트 조합")
print("   C-3: Azure AI Vision vectorizeText로 임베딩\n")

all_enriched_texts = []
all_mm_vectors = []
mm_ok, mm_fail = 0, 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="C. 텍스트→Vision 임베딩"):
    # C-1: GPT Vision으로 이미지 분석 → 텍스트 설명
    image_analysis = analyze_image_for_text(row['image_link'], row['title'], row['category'])
    
    # C-2: enriched_content 조합 (01-enrich_with_vision과 동일)
    enriched_text = create_enriched_content(row, image_analysis)
    all_enriched_texts.append(enriched_text)
    
    # C-3: Azure AI Vision으로 텍스트 임베딩
    if enriched_text and enriched_text != "정보 없음":
        mm_vec = vectorize_text_vision(enriched_text)
    else:
        mm_vec = None
    
    all_mm_vectors.append(mm_vec)
    if mm_vec:
        mm_ok += 1
    else:
        mm_fail += 1
    
    time.sleep(0.5)

df['enriched_content'] = all_enriched_texts
df['mm_vision_text_vector'] = all_mm_vectors

print(f"\n✅ Step C 완료: 성공 {mm_ok}개 / 실패 {mm_fail}개")

In [ ]:
# Step C 검증
print("📊 Step C 검증: mm_vision_text_vector 샘플")
print("=" * 80)
for i in range(min(3, len(df))):
    row = df.iloc[i]
    vec = row['mm_vision_text_vector']
    print(f"\n{i+1}. {row['title'][:45]}...")
    print(f"   enriched_content: {row['enriched_content'][:80]}...")
    print(f"   mm_vision_text_vector: {'✅ ' + str(len(vec)) + 'D' if vec else '❌'}")
print("\n" + "=" * 80)

---
## 8️⃣ 전체 처리 요약 & 인덱스 업로드

3개 Step의 결과를 확인하고 인덱스에 업로드합니다.

In [ ]:
# ── 전체 처리 요약 ──
img_count = sum(1 for v in df['image_vector'] if v is not None)
feat_count = sum(1 for v in df['structured_features'] if v is not None)
mm_count = sum(1 for v in df['mm_vision_text_vector'] if v is not None)

print("📊 전체 처리 요약")
print("=" * 60)
print(f"   image_vector:          {img_count}/{len(df)}개")
print(f"   structured_features:   {feat_count}/{len(df)}개")
print(f"   mm_vision_text_vector: {mm_count}/{len(df)}개")
print("=" * 60)

# ── 업로드할 문서 준비 ──
documents = []
for idx, row in df.iterrows():
    doc = {"id": str(row['id'])}
    has_data = False
    
    if row['image_vector'] is not None:
        doc["image_vector"] = row['image_vector']
        has_data = True
    
    if row['structured_features'] is not None:
        doc["structured_features"] = row['structured_features']
        has_data = True
    
    if row['mm_vision_text_vector'] is not None:
        doc["mm_vision_text_vector"] = row['mm_vision_text_vector']
        has_data = True
    
    if has_data:
        documents.append(doc)

print(f"\n📤 업로드 준비: {len(documents)}개 문서")

# ── 배치 업로드 ──
BATCH_SIZE = 500
total_success = 0
total_failed = 0

for i in tqdm(range(0, len(documents), BATCH_SIZE), desc="업로드"):
    batch = documents[i:i+BATCH_SIZE]
    try:
        result = search_client.merge_or_upload_documents(documents=batch)
        total_success += sum(1 for item in result if item.succeeded)
        total_failed += sum(1 for item in result if not item.succeeded)
    except Exception as e:
        print(f"⚠️  배치 업로드 실패: {str(e)[:80]}")
        total_failed += len(batch)
    time.sleep(0.5)

print(f"\n{'='*60}")
print(f"✅ 인덱스 업로드 완료!")
print(f"   성공: {total_success}개")
print(f"   실패: {total_failed}개")
print(f"{'='*60}")

---
## 9️⃣ 업로드 검증

인덱스에서 실제 문서를 조회하여 3개 필드가 정상적으로 저장되었는지 확인합니다.

In [ ]:
# 업로드 검증: 벡터 검색으로 실제 데이터 존재 확인
# (벡터 필드는 hidden=True라 get_document로 조회 불가 → 벡터 검색으로 검증)
from azure.search.documents.models import VectorizedQuery

print("🔍 업로드 검증 (벡터 검색 기반)")
print("=" * 70)

# 1) image_vector 검증 - 더미 벡터로 검색
dummy_1024 = [0.01] * 1024
results = search_client.search(
    search_text=None,
    vector_queries=[VectorizedQuery(vector=dummy_1024, k_nearest_neighbors=5, fields="image_vector")],
    select=["id", "title"],
    top=5
)
hits = list(results)
print(f"\n1️⃣ image_vector 벡터 검색: {len(hits)}개 결과")
for h in hits[:3]:
    print(f"   ID={h['id']}: {h.get('title', 'N/A')[:40]}  (score={h.get('@search.score', 0):.4f})")
print(f"   → {'✅ 데이터 존재 확인!' if hits else '❌ 데이터 없음'}")

# 2) mm_vision_text_vector 검증
results2 = search_client.search(
    search_text=None,
    vector_queries=[VectorizedQuery(vector=dummy_1024, k_nearest_neighbors=5, fields="mm_vision_text_vector")],
    select=["id", "title"],
    top=5
)
hits2 = list(results2)
print(f"\n2️⃣ mm_vision_text_vector 벡터 검색: {len(hits2)}개 결과")
for h in hits2[:3]:
    print(f"   ID={h['id']}: {h.get('title', 'N/A')[:40]}  (score={h.get('@search.score', 0):.4f})")
print(f"   → {'✅ 데이터 존재 확인!' if hits2 else '❌ 데이터 없음'}")

# 3) structured_features 검증 (일반 필드라 get_document로 조회 가능)
doc = search_client.get_document(key=str(df.iloc[0]['id']))
sf = doc.get('structured_features')
print(f"\n3️⃣ structured_features: {'✅ ' + str(sf.get('product_type', '-')) if sf else '❌ 없음'}")

print(f"\n{'='*70}")
print("💡 벡터 필드는 hidden=True로 설정되어 get_document()로 조회 불가하지만,")
print("   벡터 검색은 정상 작동합니다. (검색 성능에 영향 없음)")

In [ ]:
# 전체 인덱스 벡터 필드 현황
print("📊 인덱스 벡터 필드 현황")
print("=" * 75)

# 인덱스 스키마에서 벡터 필드 정보 조회
index_info = index_client.get_index(INDEX_NAME)

fields_info = [
    ("content_vector",        "3072D", "text-embedding-3-large", "텍스트 메타 + GPT Vision 분석"),
    ("review_vector",         "3072D", "text-embedding-3-large", "리뷰 텍스트"),
    ("image_vector",          "1024D", "Azure AI Vision",        "이미지 직접 임베딩"),
    ("mm_vision_text_vector", "1024D", "Azure AI Vision",        "GPT Vision 텍스트 → Vision 임베딩"),
]

print(f"{'필드':<25} {'차원':<8} {'hidden':<8} {'모델':<22} {'소스'}")
print("─" * 85)
for field_name, expected_dim, model, desc in fields_info:
    schema_field = next((f for f in index_info.fields if f.name == field_name), None)
    if schema_field:
        dim = getattr(schema_field, 'vector_search_dimensions', None)
        hidden = getattr(schema_field, 'hidden', None)
        print(f"  {field_name:<23} {str(dim)+'D':<8} {str(hidden):<8} {model:<22} {desc}")
    else:
        print(f"  {field_name:<23} {'N/A':<8} {'N/A':<8} {model:<22} {desc} (필드 없음)")

# structured_features
sf_field = next((f for f in index_info.fields if f.name == 'structured_features'), None)
print(f"\n  {'structured_features':<23} {'Complex':<8} {'-':<8} {'GPT-5.2':<22} 9개 피처 구조화")

print(f"\n⚠️  모든 벡터 필드가 hidden=True → get_document()로 벡터 값 조회 불가")
print(f"   단, 벡터 검색(vector search)은 정상 작동합니다.")
print(f"💡 content_vector와 mm_vision_text_vector는 같은 텍스트를 다른 모델로 임베딩한 것입니다.")

---
## ✅ 완료

### 인덱스 필드 전체 구성

| 필드 | 차원 | 임베딩 모델 | 소스 |
|------|------|------------|------|
| `content_vector` (기존) | 3072D | text-embedding-3-large | 제품메타 + GPT Vision 텍스트 |
| `review_vector` (기존) | 3072D | text-embedding-3-large | 리뷰 |
| `image_vector` (신규) | 1024D | Azure AI Vision | 이미지 픽셀 직접 |
| `mm_vision_text_vector` (신규) | 1024D | Azure AI Vision | GPT Vision 텍스트 |
| `structured_features` (신규) | Complex | GPT-5.2 | 9개 피처 JSON |

### 🧭 다음 단계

→ [02-image_based_search.ipynb](./02-image_based_search.ipynb)에서 멀티모달 검색 비교 실험

---
## 🔧 보완: image_link 필드 추가 & 데이터 업로드

인덱스 최초 생성 시 `image_link` 필드가 누락되었습니다.  
검색 결과에서 제품 이미지를 표시하려면 이 필드가 필요하므로, 여기서 추가하고 CSV 데이터를 채웁니다.

In [ ]:
# ── 1) 인덱스에 image_link 필드 추가 ──
index = index_client.get_index(INDEX_NAME)
existing_field_names = [f.name for f in index.fields]

if "image_link" not in existing_field_names:
    index.fields.append(SearchField(
        name="image_link",
        type=SearchFieldDataType.String,
        searchable=False,
        filterable=False,
    ))
    result = index_client.create_or_update_index(index)
    print(f"✅ image_link 필드 추가 완료 (총 필드: {len(result.fields)})")
else:
    print("ℹ️  image_link 필드 이미 존재")

# ── 2) CSV에서 image_link 데이터 업로드 ──
df_csv = pd.read_csv("../00-data/sample_data.csv")
documents = [{"id": str(row['id']), "image_link": row['image_link']} for _, row in df_csv.iterrows()]

result = search_client.merge_or_upload_documents(documents=documents)
success = sum(1 for r in result if r.succeeded)
failed = sum(1 for r in result if not r.succeeded)
print(f"✅ image_link 데이터 업로드: 성공 {success}개 / 실패 {failed}개")

# ── 3) 검증 ──
doc = search_client.get_document(key="1")
print(f"\n🔍 검증 ID=1: image_link = {doc.get('image_link', '없음')}")